# VMT Reduction Route Visualization

This script provides a means to visualize the routes from the VMT mode shift project for QA/QC purposes.  Start by running all the way through Part 1: Read and merge data.  This will result in a combined dataframe to support visualization that is equivalent to what is used in the feasibility analysis.  Part 2 defines the main visualization methods. In Part 3, the user can select a subset of routes using any number of queries on the main dataframe, then visualize either a selected or a randome trip from that dataframe.  


## Part 1: Read and merge data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd

import importlib
import route_mapper

from shapely.geometry import Point


In [ ]:
import math
import keyring
import itertools
from ast import literal_eval

import folium
# from folium.plugins import Legend   # having trouble on the import, using branca instead
from folium.features import GeoJsonPopup

import branca
import keyring


In [ ]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [ ]:
%%html
<style>
.rendered_html p {
    font-size: 12px;
    font-family: Times New Roman, serif;
    text-align:justify}
</style>

In [ ]:
# This is used to avoid hard-coding directories. 
# To set the directory, use the command prompt or a notebook you don't check in.  run:
# import keyring
# keyring.set_password("msp", "vmt_reduction_dir", <directory>)

# get base path for data
data_dir = keyring.get_password("msp", "vmt_reduction_dir")

In [ ]:
# read in the data
df = pd.read_csv(data_dir + "/data_processed/tbi_cleaned.csv")

In [ ]:
# # read path geopkg data
car = gpd.read_file(data_dir + "/Data_Processed/car_trips.gpkg")
bike = gpd.read_file(data_dir + "/Data_Processed/bike_lts.gpkg")
transit = gpd.read_file(data_dir + "/Data_Processed/transit-trips.gpkg")
walk = gpd.read_file(data_dir + "/Data_Processed/walk_trips.gpkg")

In [ ]:
# grab the observed routes
observed =  gpd.read_file(data_dir + "/Data_Processed/observed_locations.gpkg")

In [ ]:
# calculate duration from star/tend times
transit["start_time_dt"] = pd.to_datetime(transit["start_time"])
transit["end_time_dt"] = pd.to_datetime(transit["end_time"])

transit["duration"] = (transit["end_time_dt"] - transit["start_time_dt"]).apply(lambda x: x.seconds / 60)

In [ ]:
# create a gdf for linked trips
agg_fns = {
    "trip_id": "first",
    "leg_index": "count",
    "start_time": "first",
    "end_time": "last",
    "origin_stop_id": "first", # throw away
    "origin_stop_name": "first",
    "destination_stop_id": "first",
    "destination_stop_name": "first",
    "route_id": "first", 
    "route_short_name": "first",
    "route_long_name": "first",
    "route_type": "first", 
    "leg_type": "first", # end throw away
    "start_time_dt": "first",
    "end_time_dt": "last",
    "duration": "sum"
}

transit_grouped = transit.dissolve(by="trip_id", aggfunc=agg_fns) # merges geometries in addition to aggregating the rest of the columns

## Step 2: Define viz functions

In [ ]:
# Categorical color scheme for various travel modes
colorMap = {
    'bike': '#00008B',      # darkblue
    'walk': '#006400',      # darkgreen
    'transit': '#4B0082',   # indigo
    'car': '#8B4513',       # saddlebrown
    'location':'#E46707'    # orange
}

# Object for storing attributes used to filter datasets
attributeMap = {
    'Mode of route': 'mode',
    'Origin purpose': 'o_purpose',
    'Destination purpose': 'd_purpose',
    'Number of travelers': 'num_travelers',
    'Distance': 'distance',
    'Speed (MPH)': 'speed_mph',
    'VMT': 'vmt',
    'Age': 'age'
}

# conditions for evaluating query expressions
conditions = ['==', '!=', '<', '<=', '>', '>=']

In [ ]:
def add_car_route(car_gdb, map, trip_id): 
    
    # car mode
    car_trip = car_gdb[car_gdb['trip_id']==trip_id]
    fields=['distance_meters', 'duration_seconds','weight']
    aliases=['Distance (meters)', 'Duration (sec)', 'Generalized Cost (sec)']
    car_tooltip=folium.features.GeoJsonTooltip(fields=fields, aliases=aliases)
    car_popup=folium.features.GeoJsonPopup(fields=fields, aliases=aliases)
    folium.GeoJson(car_trip, 
                   name='Car route', 
                   tooltip=car_tooltip, 
                   popup=car_popup, 
                   style_function=lambda feature: {'color':colorMap['car']}).add_to(map)

In [ ]:
def add_walk_route(walk_gdb, map, trip_id):

    # walk mode    
    walk_trip = walk_gdb[walk_gdb['trip_id']==trip_id]
    walk_tooltip=folium.features.GeoJsonTooltip(fields=['distance_meters', 
                                                   'duration_seconds', 
                                                   'weight'],
                                           aliases=['Distance (meters)', 
                                                    'Duration (sec)', 
                                                    'Generalized Cost (sec)'])
    folium.GeoJson(walk_trip, 
                   name='Walk route', 
                   tooltip=walk_tooltip, 
                   style_function=lambda feature: {'color':colorMap['walk']}).add_to(map)
    

In [ ]:
def add_bike_route(bike_gdb, map, trip_id):
    
    # bike mode
    bike_trip = bike_gdb[bike_gdb['trip_id']==trip_id]
    bike_tooltip=folium.features.GeoJsonTooltip(fields=['distance_meters', 
                                                   'duration_seconds', 
                                                   'weight'],
                                           aliases=['Distance (meters)', 
                                                    'Duration (sec)', 
                                                    'Generalized Cost (sec)'])
    folium.GeoJson(bike_trip, 
                   name='Bike route', 
                   tooltip=bike_tooltip, 
                   style_function=lambda feature: {'color':colorMap['bike']}).add_to(map)

In [ ]:
def add_transit_route(transit_gdb, map, trip_id): 
    
    # transit mode
    transit_legs = transit_gdb[transit_gdb['trip_id']==trip_id]
    transit_leg_tooltip = {}
    for i, leg in transit_legs.iterrows(): 
        # plot the route
        transit_leg_tooltip[i] = folium.features.GeoJsonTooltip(fields=['leg_type', 
                                                                   'route_short_name', 
                                                                   'start_time_dt', 
                                                                   'end_time_dt', 
                                                                   'duration'],
                                                          aliases=['Leg Type', 
                                                                   'Start Time', 
                                                                   'End Time', 
                                                                   'Route Short Name', 
                                                                   'Duration (sec)'])
        
        folium.GeoJson(leg.geometry, 
                       name='Transit Leg', 
                       #tooltip=transit_leg_tooltip[i], 
                       style_function=lambda feature: {'color':colorMap['transit']}).add_to(map)
        
    
        # plot the first point
        folium.CircleMarker(location=[leg.geometry.coords[0][1], leg.geometry.coords[0][0]], 
                           radius=5, 
                           fill=True, 
                           color=colorMap['transit']).add_to(map)
        
        # plot the last point
        folium.CircleMarker(location=[leg.geometry.coords[-1][1], leg.geometry.coords[-1][0]],   
                           radius=5, 
                           fill=True, 
                           color=colorMap['transit']).add_to(map)
        

In [ ]:
def add_observed_route(observed_gdb, map, trip_id): 
    
    # observed locations
    observed_points = observed[observed['trip_id'].apply(lambda x : x in trip_leg_ids)]
    for i, point in observed_points.iterrows():         
        folium.CircleMarker(location=[point.geometry.y, point.geometry.x], 
                           radius=5, 
                           fill=True, 
                           color=colorMap['location']).add_to(map)
    

In [ ]:

def map_trip(trip_id="Random"):
    
    if trip_id=="Random":
        trip_id = df.sample(n=1)['trip_id'].iloc[0]
    trip = df[df['trip_id']==trip_id]
    
    # create a list of each individual lev
    trip_leg_ids_string = trip_id.replace('[','').replace(']','').split(',')
    trip_leg_ids = [int(i) for i in trip_leg_ids_string]
        
    # tooltip current broken when adding as arugment below in folium.GeoJson
    tooltip=folium.features.GeoJsonTooltip(fields=['mode', 'o_purpose', 'd_purpose', 'num_travelers', 'distance', 'speed_mph', 'vmt', 'age'],
                                         aliases=['Mode', 'O purpose', 'D purpose', '# of travelers', 'Distance', 'Speed (mph)', 'VMT', 'Age'])
        
    # instantiate a global Leaflet map object using Folium
    map = folium.Map(location=[44.9778,-93.2650], tiles="CartoDB positron", zoom_start=10)
    
    add_car_route(car_gdb, m, trip_id)
    add_walk_route(walk_gdb, m, trip_id)
    add_bike_route(bike_gdb, m, trip_id)
    add_transit_route(transit_gdb, m, trip_id)
    add_observed_route(observed_gdb, m, trip_id)

    # create html string for legend
    legend_html = '''
    <div style="position: fixed; 
        bottom: 50px; right: 50px; width: 150px; height: 125px; 
        border:2px solid grey; z-index: 9999; font-size:14px;">
        &nbsp;<b>Legend</b><br>
        &nbsp;<i class="fa fa-circle" style="color:#006400"></i>&nbsp;Walking<br>
        &nbsp;<i class="fa fa-circle" style="color:#00008B"></i>&nbsp;Biking<br>
        &nbsp;<i class="fa fa-circle" style="color:#8B4513"></i>&nbsp;Driving<br>
        &nbsp;<i class="fa fa-circle" style="color:#4B0082"></i>&nbsp;Transit<br>
        &nbsp;<i class="fa fa-circle" style="color:#E46707"></i>&nbsp;Observed
    </div>
    '''
    
    # use branca to add legend to the map
    legend = branca.element.Element(legend_html)
    map.get_root().html.add_child(legend)

    route_bounds = bike_trip.bounds
    map.fit_bounds([[route_bounds['miny'].values[0], 
                     route_bounds['minx'].values[0]], 
                    [route_bounds['maxy'].values[0], 
                     route_bounds['maxx'].values[0]]])

    # add a legend to the map
    folium.LayerControl().add_to(map)
    return map


In [ ]:
transit_subsample = df[df['mode']=="Transit"]

In [ ]:
bike_subsample = df[df['mode']=="Bike/Scooter"]
len(bike_subsample)

In [ ]:
# use one of the mode routes to select a random trip ID and create a map
trip_id = transit_subsample.sample(n=1)['trip_id'].iloc[0]
map = map_trip(trip_id)
map

In [ ]:
%load_ereloaxt autoreload
%autoreload 2

In [ ]:
# reload is only needed if we make changes to the module
importlib.reload(route_mapper)
mapper = route_mapper.RouteMapper(df, car, walk, bike, transit, observed)

In [ ]:
trip_id = transit_subsample.sample(n=1)['trip_id'].iloc[0]

In [ ]:
mapper.map_trip(trip_id)